# NB2 – Kiểm tra trùng lặp giữa các split (R2-C2)
**Runtime:** CPU · High-RAM (càng nhiều nhân càng nhanh)  ·  **Thời gian ước tính:** 1,5–3 giờ

Notebook tự chứa: code được nhúng sẵn (các ô `%%writefile`), không cần upload file.
Có thể chạy **song song** với các notebook khác. Chọn *Runtime → Run all*.
Kết quả được chép vào `MyDrive/PLOS_revision/NB2_CPU_duplicates/`.

Gom file dự đoán → nối các bộ dữ liệu → trùng lặp chính xác + gần trùng (MinHash, Jaccard ≥ 0,8) → chấm lại điểm trên tập test đã loại trùng.

In [ ]:
# 1) Drive (kết quả được chép vào đây) + 2) đăng nhập Hugging Face (tài khoản BaoNhan)
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
RESULT_DIR = "/content/drive/MyDrive/PLOS_revision/NB2_CPU_duplicates"
os.makedirs(RESULT_DIR, exist_ok=True)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")      # Colab secret tên HF_TOKEN (nếu có)
except Exception:
    from huggingface_hub import login
    login()                                                 # nếu chưa có secret: dán token vào ô hiện ra
from huggingface_hub import whoami
try:
    print("Đã đăng nhập Hugging Face:", whoami()["name"])
except Exception:
    raise SystemExit("CHƯA ĐĂNG NHẬP Hugging Face. Bấm biểu tượng chìa khóa (Secrets) bên trái, "
                     "bật 'Notebook access' cho HF_TOKEN, rồi chạy lại ô này.")
!pip -q install datasets pyarrow scikit-learn

## Code (nhúng sẵn – không cần sửa)

In [ ]:
%%writefile common.py
"""Shared configuration for the PLOS ONE revision analyses (PONE-D-26-38656).

Every constant below is taken from the manuscript or from the Hugging Face
repositories listed by step0_inspect.py. Nothing here is estimated.
"""
from pathlib import Path

SEEDS = [42, 22, 202]
FIG_SEED_ORDER = [22, 42, 202]          # panel order of the original Fig 5
MAX_LEN = 256                           # model-specific token budget
N_CLASSES = 10

# Model display name -> (HF repo, short key used in folder/file names)
MODELS = {
    "WikiBERT":   ("BaoNhan/WikiBERT_Vietnamese_News_Classification", "wikibert"),
    "PhoBERT":    ("BaoNhan/PhoBERT_Vietnamese_News_Classification", "phobert"),
    "mBERT":      ("BaoNhan/mBERT_Vietnamese_News_Classification", "mbert"),
    "DistilBERT": ("BaoNhan/DistilBERT_Vietnamese_News_Classification", "distilbert"),
    "XLM-R":      ("BaoNhan/XLMR_Vietnamese_News_Classification", "xlmr"),
    "CafeBERT":   (None, "cafebert"),   # fine-tuned CafeBERT runs are not on the Hub
    "viBERT":     ("BaoNhan/viBERT_Vietnamese_News_Classification", "vibert"),
    "vELECTRA":   ("BaoNhan/vELECTRA_Vietnamese_News_Classification", "velectra"),
    "ViSoBERT":   ("BaoNhan/ViSoBERT_Vietnamese_News_Classification", "visobert"),
}
MODEL_ORDER = list(MODELS)  # same order as Table 2
# Public pre-trained checkpoints that were fine-tuned (verified to exist on the Hub;
# step4 also checks their sizes against the released fine-tuned configs).
BASE_CHECKPOINTS = {
    "WikiBERT":   "TurkuNLP/wikibert-base-vi-cased",
    "PhoBERT":    "vinai/phobert-base",
    "mBERT":      "google-bert/bert-base-multilingual-cased",
    "DistilBERT": "distilbert/distilbert-base-multilingual-cased",
    "XLM-R":      "FacebookAI/xlm-roberta-base",
    "CafeBERT":   "uitnlp/CafeBERT",
    "viBERT":     "FPTAI/vibert-base-cased",
    "vELECTRA":   "FPTAI/velectra-base-discriminator-cased",
    "ViSoBERT":   "uitnlp/visobert",
}
# Models whose seed-specific TCC was not retained: Fig 5 uses the Table 2 mean TCC
# for every seed of these models and the caption states this explicitly.
MEAN_TCC_FALLBACK = ["PhoBERT", "CafeBERT"]

# Category indices used by all prediction files (manuscript, Results section)
LABEL_NAMES = ["Education", "Entertainment", "Business", "Law", "Health",
               "World", "Sports", "Politics", "Automotive", "Lifestyle"]
# Test-set row totals per index, as reported in the manuscript
TEST_COUNTS = [6090, 9542, 12350, 6992, 9766, 13673, 12423, 13490, 4844, 5500]
N_TEST = 94670

# Table 2 of the submitted manuscript: mean (population SD) over seeds
TABLE2 = {  # loss, acc, P, R, F1, AUC, TCC
    "WikiBERT":   [0.20330, 0.93016, 0.92377, 0.92264, 0.92302, 0.99635, 3077.08],
    "PhoBERT":    [0.14761, 0.94779, 0.94124, 0.94375, 0.94241, 0.99794, 3360.89],
    "mBERT":      [0.16421, 0.94270, 0.93707, 0.93660, 0.93674, 0.99750, 3181.59],
    "DistilBERT": [0.17685, 0.93793, 0.93145, 0.93213, 0.93167, 0.99713, 1554.86],
    "XLM-R":      [0.15650, 0.94488, 0.93907, 0.94011, 0.93953, 0.99770, 3099.05],
    "CafeBERT":   [0.14729, 0.94916, 0.94391, 0.94445, 0.94413, 0.99800, 9335.51],
    "viBERT":     [0.20287, 0.92903, 0.92226, 0.92185, 0.92190, 0.99629, 3065.10],
    "vELECTRA":   [0.18201, 0.93791, 0.93174, 0.93189, 0.93161, 0.99710, 3192.59],
    "ViSoBERT":   [0.21663, 0.92617, 0.91819, 0.92075, 0.91934, 0.99604, 3201.82],
}
TABLE2_TCC_SD = {"WikiBERT": 3.46492, "PhoBERT": 4.53214, "mBERT": 1.24008,
                 "DistilBERT": 0.99322, "XLM-R": 1.41930, "CafeBERT": 148.46973,
                 "viBERT": 1.53178, "vELECTRA": 1.57443, "ViSoBERT": 0.44646}

COLORS = {"WikiBERT": "#4C78A8", "PhoBERT": "#F58518", "mBERT": "#54A24B",
          "DistilBERT": "#E45756", "XLM-R": "#B279A2", "CafeBERT": "#72B7B2",
          "viBERT": "#FF9DA6", "vELECTRA": "#9D755D", "ViSoBERT": "#BAB0AC"}

DATA = Path("revision_data")      # created by step1
OUT = Path("revision_results")    # created by step2/step3


def pred_path(model, seed):
    return DATA / "preds" / f"{model}_seed{seed}.npz"

In [ ]:
%%writefile step1_collect.py
"""STEP 1 - Download the released predictions/metrics and assemble them.

Runs on CPU. Requires `huggingface_hub`, `datasets`, `pandas`, `numpy`.
Output folder: revision_data/
  preds/<Model>_seed<s>.npz   true, pred, prob (aligned row by row)
  test_text.pkl           row, id, content of the 94,670 test articles
  metrics/<Model>.json        raw per-seed metric JSON(s) as released
  dataset_meta.pkl        label/url/source of BaoNhan/Vietnam_News_3_Sources
  step1_report.txt            every check performed and its outcome

The script STOPS with an explanation if any consistency check fails.
"""
import json
import sys

import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download

from common import (DATA, MODELS, N_TEST, SEEDS, TEST_COUNTS,
                    pred_path)

api = HfApi()
(DATA / "preds").mkdir(parents=True, exist_ok=True)
(DATA / "metrics").mkdir(parents=True, exist_ok=True)
report = []


def log(msg):
    print(msg)
    report.append(msg)


def fail(msg):
    log("STOP: " + msg)
    (DATA / "step1_report.txt").write_text("\n".join(report), encoding="utf-8")
    sys.exit(1)


# ---------------------------------------------------------------- locate files
locations = {}  # (model, seed) -> (repo, csv path)
metric_files = {}  # model -> list of (repo, path)
import os
ONLY = [m.strip() for m in os.environ.get("ONLY_MODELS", "").split(",") if m.strip()]
for model, (repo, key) in MODELS.items():
    if repo is None or (ONLY and model not in ONLY):
        continue
    files = api.list_repo_files(repo)
    for s in SEEDS:
        p = f"results_{key}/predictions_seed_{s}.csv"
        if p not in files:
            fail(f"{repo} has no {p}")
        locations[(model, s)] = (repo, p)
    metric_files[model] = [(repo, f) for f in files
                           if f.startswith(f"results_{key}/") and f.endswith(".json")]

metric_files["CafeBERT"] = []
log("CafeBERT: fine-tuned predictions are not released on the Hub - skipped")

# ---------------------------------------------------------------- predictions
PROB_COLS = [f"prob_class_{k}" for k in range(10)]
ref_true, ref_id, text_saved = None, None, False
for (model, s), (repo, path) in sorted(locations.items()):
    local = hf_hub_download(repo, path)
    header = pd.read_csv(local, nrows=0).columns.tolist()
    use = [c for c in header if c != "content"]
    df = pd.read_csv(local, usecols=use)
    if len(df) != N_TEST:
        fail(f"{model} seed {s}: {len(df)} rows, expected {N_TEST}")
    for c in ["true_label_0_9", "predicted_label_0_9"] + PROB_COLS:
        if c not in df.columns:
            fail(f"{model} seed {s}: column {c} missing; columns = {header}")
    true = df["true_label_0_9"].to_numpy(np.int8)
    pred = df["predicted_label_0_9"].to_numpy(np.int8)
    prob = df[PROB_COLS].to_numpy(np.float32)
    if ref_true is None:
        ref_true = true
        counts = np.bincount(true, minlength=10).tolist()
        if counts != TEST_COUNTS:
            fail(f"class counts {counts} differ from manuscript {TEST_COUNTS}")
        log(f"class counts match the manuscript: {counts}")
    elif not np.array_equal(true, ref_true):
        fail(f"{model} seed {s}: true-label order differs from the reference file")
    if "id" in df.columns:
        ids = df["id"].to_numpy()
        if ref_id is None:
            ref_id = ids
        elif not np.array_equal(ids, ref_id):
            fail(f"{model} seed {s}: id order differs from the reference file")
        if not text_saved:
            txt = pd.read_csv(local, usecols=["id", "content"])
            txt.insert(0, "row", np.arange(len(txt)))
            txt.to_pickle(DATA / "test_text.pkl")
            text_saved = True
            log(f"test text saved from {repo}/{path}")
    np.savez_compressed(pred_path(model, s), true=true, pred=pred, prob=prob)
    log(f"{model:10s} seed {s:3d}: OK  acc={np.mean(true == pred):.5f}  "
        f"id column={'id' in df.columns}")

missing = [(m, s) for m in MODELS for s in SEEDS if (m, s) not in locations and (not ONLY or m in ONLY)]
log(f"prediction files missing: {missing if missing else 'none'}")

# ---------------------------------------------------------------- metrics json
for model, lst in metric_files.items():
    merged = {}
    for repo, f in lst:
        merged[f"{repo}/{f}"] = json.load(open(hf_hub_download(repo, f), encoding="utf-8"))
    (DATA / "metrics" / f"{model}.json").write_text(
        json.dumps(merged, indent=2, ensure_ascii=False), encoding="utf-8")
    log(f"{model}: {len(lst)} metric JSON file(s) saved")

# ---------------------------------------------------------------- dataset meta
try:
    from datasets import load_dataset
    ds = load_dataset("BaoNhan/Vietnam_News_3_Sources")["train"].to_pandas()
    ds.insert(0, "row", np.arange(len(ds)))
    ds.to_pickle(DATA / "dataset_meta.pkl")
    log(f"dataset metadata saved: {len(ds)} rows, columns {list(ds.columns)}")
    if ref_id is not None:
        log(f"test id examples: {ref_id[:5].tolist()}  dtype={ref_id.dtype}")
except Exception as e:  # noqa: BLE001
    log(f"dataset metadata NOT saved: {e!r}")

(DATA / "step1_report.txt").write_text("\n".join(report), encoding="utf-8")
print("\nDone. Send revision_data/step1_report.txt.")

In [ ]:
%%writefile step1b_index.py
"""STEP 1b - Link the three private datasets row by row (CPU, high-RAM runtime).

  BaoNhan/Vietnamese_News_Cleaned_3_Sources_Split  label(int), content  train/dev/test
  BaoNhan/Vietnamese_News_PhoBERT_Segmented        same rows, word-segmented content
  BaoNhan/Vietnam_News_3_Sources_Plus              url, label(str), content, source,
                                                   day, month, year (one split)

Output revision_data/corpus_index.pkl - one row per Cleaned article:
  split, pos, label (0-9 index used by the prediction files), source, year, url,
  plus_row (row in the Plus dataset, -1 if not found), h_exact, h_norm (content hashes)
and revision_data/step1b_report.txt. Every link is verified, nothing is assumed:
  * Cleaned and Segmented must have identical labels at every (split, pos)
  * the Cleaned test labels must equal the prediction-file labels (offset detected)
  * Plus rows are matched to Cleaned rows by an exact hash of the content
    (a whitespace-normalised hash is used only for rows without an exact match)
  * the category names implied by the match are checked against the manuscript
"""
import hashlib
import re
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download

from common import DATA, LABEL_NAMES, MODEL_ORDER, SEEDS, pred_path

CLEAN = "BaoNhan/Vietnamese_News_Cleaned_3_Sources_Split"
SEG = "BaoNhan/Vietnamese_News_PhoBERT_Segmented"
PLUS = "BaoNhan/Vietnam_News_3_Sources_Plus"
SPLIT_SIZES = {"train": 757356, "dev": 94670, "test": 94670}
report = []
WS = re.compile(r"\s+")


def log(msg=""):
    print(msg)
    report.append(str(msg))


def stop(msg):
    log("STOP: " + msg)
    (DATA / "step1b_report.txt").write_text("\n".join(report), encoding="utf-8")
    sys.exit(1)


def h_exact(t):
    return hashlib.blake2b((t or "").encode("utf-8"), digest_size=8).hexdigest()


def h_norm(t):
    return h_exact(WS.sub(" ", t or "").strip())


def read_repo(repo, prefix, columns):
    """Stream a split's parquet shards; content is reduced to hashes immediately."""
    files = sorted(f for f in HfApi().list_repo_files(repo, repo_type="dataset")
                   if f.startswith(f"data/{prefix}-") and f.endswith(".parquet"))
    out = []
    for f in files:
        pf = pq.ParquetFile(hf_hub_download(repo, f, repo_type="dataset"))
        for b in pf.iter_batches(batch_size=20000, columns=columns):
            d = b.to_pandas()
            if "content" in d:
                d["h_exact"] = d["content"].map(h_exact)
                d["h_norm"] = d["content"].map(h_norm)
                d = d.drop(columns="content")
            out.append(d)
    return pd.concat(out, ignore_index=True)


# -------------------------------------------------------- Cleaned and Segmented
parts = []
for split, n in SPLIT_SIZES.items():
    c = read_repo(CLEAN, split, ["label", "content"])
    s = read_repo(SEG, split, ["label"])
    if len(c) != n or len(s) != n:
        stop(f"{split}: Cleaned {len(c)} / Segmented {len(s)} rows, expected {n}")
    if not np.array_equal(c["label"].to_numpy(), s["label"].to_numpy()):
        stop(f"{split}: Cleaned and Segmented labels differ - rows are not aligned")
    c.insert(0, "split", split)
    c.insert(1, "pos", np.arange(n))
    parts.append(c)
    log(f"{split}: {n} rows, Cleaned/Segmented labels identical at every position; "
        f"label range {c['label'].min()}..{c['label'].max()}")
idx = pd.concat(parts, ignore_index=True)

# -------------------------------------------------------- label offset vs predictions
ref = next((pred_path(m, s) for m in MODEL_ORDER for s in SEEDS if pred_path(m, s).exists()), None)
if ref is None:
    stop("no prediction file found - run step1_collect.py first")
true = np.load(ref)["true"].astype(int)
test_lab = idx.loc[idx.split == "test", "label"].to_numpy()
offset = next((k for k in (0, 1) if np.array_equal(test_lab - k, true)), None)
if offset is None:
    stop("Cleaned test labels do not equal the prediction-file labels (offset 0 or 1)")
idx["label"] = idx["label"] - offset
log(f"Cleaned test split == prediction-file order; dataset label = index + {offset}")

# -------------------------------------------------------- Plus metadata
plus = read_repo(PLUS, "train", ["url", "label", "content", "source", "year"])
plus.insert(0, "plus_row", np.arange(len(plus)))
log(f"Plus: {len(plus)} rows; sources {plus.source.value_counts().to_dict()}")
for key in ("h_exact", "h_norm"):
    d = plus[key].duplicated(keep=False).sum()
    log(f"Plus rows sharing a {key} with another row: {d}")

m = idx.merge(plus.drop_duplicates("h_exact")[["h_exact", "plus_row"]], on="h_exact", how="left")
miss = m.plus_row.isna()
if miss.any():
    alt = plus.drop_duplicates("h_norm")[["h_norm", "plus_row"]].rename(columns={"plus_row": "pr2"})
    m = m.merge(alt, on="h_norm", how="left")
    m.loc[miss, "plus_row"] = m.loc[miss, "pr2"]
    m = m.drop(columns="pr2")
m["plus_row"] = m["plus_row"].fillna(-1).astype(int)
log(f"Cleaned rows matched to Plus: {(m.plus_row >= 0).mean():.4%} "
    f"(exact {(~miss).mean():.4%})")
meta = plus.set_index("plus_row")
ok = m.plus_row >= 0
for col in ("url", "source", "year"):
    m.loc[ok, col] = meta.loc[m.loc[ok, "plus_row"], col].to_numpy()
m.loc[ok, "plus_label"] = meta.loc[m.loc[ok, "plus_row"], "label"].to_numpy()

# -------------------------------------------------------- category-name check
ct = pd.crosstab(m.loc[ok, "label"], m.loc[ok, "plus_label"])
implied = ct.idxmax(axis=1).reindex(range(10)).tolist()
purity = (ct.max(axis=1) / ct.sum(axis=1)).round(4).tolist()
log(f"category name implied by Plus for index 0..9: {implied}")
log(f"share of rows agreeing with that name: {purity}")
log(f"manuscript names for index 0..9:          {LABEL_NAMES}")
log("NAMES AGREE" if implied == LABEL_NAMES else "NAMES DIFFER - report this before using them")
for split in SPLIT_SIZES:
    s = m[m.split == split]
    log(f"{split}: years {s.year.value_counts().sort_index().to_dict()}")

m.to_pickle(DATA / "corpus_index.pkl")
(DATA / "step1b_report.txt").write_text("\n".join(report), encoding="utf-8")
print("Done.")

In [ ]:
%%writefile step3b_duplicates.py
"""STEP 3b - Cross-partition duplicate audit (Reviewer #2, Concern 2). CPU, high-RAM.

Exact duplicates : identical content hash (raw, and whitespace-normalised).
Near duplicates  : MinHash-LSH on lower-cased word 5-gram shingles,
                   estimated Jaccard similarity >= THRESHOLD (default 0.8).
For every dev/test article we record whether it has an exact or near duplicate
in train (and test vs dev). The released predictions are then re-scored on the
test articles with no duplicate in train or dev.

Self-contained MinHash-LSH (numpy only).
Outputs (revision_results/): duplicates_summary.csv, duplicates_test_flags.pkl,
duplicates_rescored.csv, step3b_report.txt
"""
import os
import re
import zlib
from multiprocessing import Pool

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download
from sklearn.metrics import precision_recall_fscore_support

from common import DATA, MODEL_ORDER, OUT, SEEDS, pred_path

CLEAN = "BaoNhan/Vietnamese_News_Cleaned_3_Sources_Split"
THRESHOLD = float(os.environ.get("NEARDUP_THRESHOLD", 0.8))
NUM_PERM, BANDS = 128, 16          # 16 bands x 8 rows: candidate recall high at J >= 0.8
ROWS = NUM_PERM // BANDS
SHINGLE = 5
MAX_CAND = 500                      # candidates checked per query and band
OUT.mkdir(parents=True, exist_ok=True)
report = []
WS = re.compile(r"\s+")
_P = np.uint64((1 << 31) - 1)
_rng = np.random.default_rng(1)
_A = _rng.integers(1, (1 << 31) - 1, NUM_PERM, dtype=np.uint64)
_B = _rng.integers(0, (1 << 31) - 1, NUM_PERM, dtype=np.uint64)


def log(msg=""):
    print(msg, flush=True)
    report.append(str(msg))


def signature(text):
    """Self-contained MinHash (no external library): deterministic across processes."""
    words = WS.sub(" ", (text or "").lower()).strip().split(" ")
    sh = {" ".join(words[i:i + SHINGLE]) for i in range(max(1, len(words) - SHINGLE + 1))}
    x = np.fromiter((zlib.crc32(t.encode("utf-8")) for t in sh), dtype=np.uint64, count=len(sh))
    return ((np.outer(x, _A) + _B) % _P).min(axis=0).astype(np.uint32)


def signatures(split):
    files = sorted(f for f in HfApi().list_repo_files(CLEAN, repo_type="dataset")
                   if f.startswith(f"data/{split}-") and f.endswith(".parquet"))
    sigs = []
    with Pool(os.cpu_count()) as pool:
        for f in files:
            pf = pq.ParquetFile(hf_hub_download(CLEAN, f, repo_type="dataset"))
            for b in pf.iter_batches(batch_size=20000, columns=["content"]):
                sigs += pool.map(signature, b.column("content").to_pylist(), chunksize=500)
            log(f"  {split}: {len(sigs)} signatures")
    return np.vstack(sigs)


def band_keys(sig):
    mult = np.uint64(0x9E3779B97F4A7C15)
    keys = np.zeros((len(sig), BANDS), dtype=np.uint64)
    s = sig.astype(np.uint64)
    with np.errstate(over="ignore"):
        for b in range(BANDS):
            k = np.zeros(len(sig), dtype=np.uint64)
            for r in range(ROWS):
                k = k * mult + s[:, b * ROWS + r]
            keys[:, b] = k
    return keys


def near_flags(q_sig, r_sig):
    """True for each query with an estimated Jaccard >= THRESHOLD to any reference."""
    qk, rk = band_keys(q_sig), band_keys(r_sig)
    flag = np.zeros(len(q_sig), dtype=bool)
    for b in range(BANDS):
        order = np.argsort(rk[:, b], kind="stable")
        sorted_keys = rk[order, b]
        lo = np.searchsorted(sorted_keys, qk[:, b], "left")
        hi = np.searchsorted(sorted_keys, qk[:, b], "right")
        for i in np.where((hi > lo) & ~flag)[0]:
            cand = order[lo[i]:min(hi[i], lo[i] + MAX_CAND)]
            if (np.mean(r_sig[cand] == q_sig[i], axis=1) >= THRESHOLD).any():
                flag[i] = True
    return flag


idx = pd.read_pickle(DATA / "corpus_index.pkl")

# ------------------------------------------------------------------ exact
flags = {}
for key in ("h_exact", "h_norm"):
    tr = set(idx.loc[idx.split == "train", key])
    dv = set(idx.loc[idx.split == "dev", key])
    for split in ("dev", "test"):
        s = idx[idx.split == split]
        flags[(split, key, "train")] = s[key].isin(tr).to_numpy()
        if split == "test":
            flags[(split, key, "dev")] = s[key].isin(dv).to_numpy()

# ------------------------------------------------------------------ near
log(f"MinHash: {NUM_PERM} permutations, {BANDS} bands x {ROWS} rows, "
    f"{SHINGLE}-word shingles, threshold={THRESHOLD}")
sig = {s: signatures(s) for s in ("train", "dev", "test")}
flags[("dev", "near", "train")] = near_flags(sig["dev"], sig["train"])
flags[("test", "near", "train")] = near_flags(sig["test"], sig["train"])
flags[("test", "near", "dev")] = near_flags(sig["test"], sig["dev"])

rows = [dict(query_split=q, kind=k, reference_split=r, n_flagged=int(v.sum()),
             pct=100 * v.mean()) for (q, k, r), v in flags.items()]
summ = pd.DataFrame(rows)
summ.to_csv(OUT / "duplicates_summary.csv", index=False)
log(summ.round(3).to_string(index=False))

test_dup = np.zeros(int((idx.split == "test").sum()), dtype=bool)
for (q, k, r), v in flags.items():
    if q == "test":
        test_dup |= v
pd.DataFrame({k[1] + "_in_" + k[2]: v for k, v in flags.items() if k[0] == "test"}
             ).assign(any_duplicate=test_dup).to_pickle(OUT / "duplicates_test_flags.pkl")
log(f"test articles with any exact or near duplicate in train/dev: "
    f"{test_dup.sum()} ({100*test_dup.mean():.2f}%)")

# ------------------------------------------------------------------ rescore
res = []
for m in MODEL_ORDER:
    if not all(pred_path(m, s).exists() for s in SEEDS):
        log(f"{m}: predictions missing, not rescored")
        continue
    for name, mask in [("full_test", np.ones_like(test_dup)), ("deduplicated_test", ~test_dup)]:
        f1, acc = [], []
        for s in SEEDS:
            r = np.load(pred_path(m, s))
            y, p = r["true"][mask], r["pred"][mask]
            f1.append(precision_recall_fscore_support(y, p, average="macro", zero_division=0)[2])
            acc.append(np.mean(y == p))
        res.append(dict(model=m, subset=name, n=int(mask.sum()), macro_f1_mean=np.mean(f1),
                        macro_f1_sd=np.std(f1), acc_mean=np.mean(acc), acc_sd=np.std(acc)))
res = pd.DataFrame(res)
res.to_csv(OUT / "duplicates_rescored.csv", index=False)
log(res.pivot(index="model", columns="subset", values="macro_f1_mean").mul(100).round(3).to_string())
(OUT / "step3b_report.txt").write_text("\n".join(report), encoding="utf-8")
print("Done.")

## Dùng lại kết quả cũ trên Drive

In [ ]:
# Dùng lại kết quả của lần chạy trước trên Drive (nếu có) để bỏ qua các bước đã xong
import shutil, glob, os
for d in ["revision_data", "revision_data/preds", "revision_results"]:
    os.makedirs(d, exist_ok=True)
    for f in glob.glob(f"{RESULT_DIR}/{d}/*"):
        if os.path.isfile(f) and not os.path.exists(f"{d}/{os.path.basename(f)}"):
            shutil.copy(f, f"{d}/")
have = lambda p: os.path.exists(p)
print("preds:", len(glob.glob("revision_data/preds/*.npz")), "| corpus_index:", have("revision_data/corpus_index.pkl"),
      "| step2:", have("revision_results/step2_report.txt"))

## Chạy

In [ ]:
if len(glob.glob("revision_data/preds/*.npz")) < 24:
    !python step1_collect.py    # cần dự đoán của cả 8 mô hình để chấm lại trên tập test đã loại trùng
if not have("revision_data/corpus_index.pkl"):
    !python step1b_index.py
!python step3b_duplicates.py

## Lưu kết quả

In [ ]:
# Chép kết quả sang Drive và tạo file zip để tải về
import shutil, glob
for d in ["revision_results", "revision_data"]:
    for f in glob.glob(f"{d}/*.txt") + glob.glob(f"{d}/*.csv") + glob.glob(f"{d}/*.tex") + \
             glob.glob(f"{d}/*.pdf") + glob.glob(f"{d}/*.tif") + glob.glob(f"{d}/*.pkl"):
        if os.path.getsize(f) < 500e6:
            os.makedirs(f"{RESULT_DIR}/{d}", exist_ok=True)
            shutil.copy(f, f"{RESULT_DIR}/{d}/")
os.makedirs(f"{RESULT_DIR}/revision_data/preds", exist_ok=True)
for f in glob.glob("revision_data/preds/*.npz"):
    shutil.copy(f, f"{RESULT_DIR}/revision_data/preds/")
print("Đã chép vào", RESULT_DIR)